> This notebook is partially adapted from materials in the repository [*BioimageCourseGIMM*](https://github.com/brunomsaraiva/BioimageCourseGIMM) by Bruno Manuel Santos Saraiva.

# Deep Learning use case - Cellpose

[GitHub repo](https://github.com/MouseLand/cellpose)    ---    [Cellpose 3.x docs](https://cellpose.readthedocs.io/en/v3.1.1.1/index.html) --- 
[Cellpose 4.x (cellpose-sam) docs](https://cellpose.readthedocs.io/en/latest/)

Cellpose GUI - differences between versions

<img src="./assets/cellpose_gui.png" width="500">

Test cellposeSAM without installation via [HuggingFace](https://huggingface.co/spaces/mouseland/cellpose).

---

## Cellpose 3.1.1.1 GUI

<div style="border-left: 4px solid #4CAF50; padding-left: 1em;">

**TASK**: Use Cellpose GUI to test the pre-trained built-in models on the example images.

- Open a new terminal
- Activate the course environment (`conda activate <env_name>`)
- Start the Cellpose GUI by running command: `cellpose`
- Explore the GUI together
- Test the pre-trained models on the example images in the folder *cellpose_example*

</div>

Example data sources:
- [HPA cell image segmentation dataset](https://bioimage.io/#/?type=dataset&id=hpa%2Fhpa-cell-image-segmentation-dataset)
- [Stardist dataset](https://bioimage.io/#/?type=all&tags=stardist&id=zero%2Fdataset_stardist_2d_zerocostdl4mic_2d)
- [Deepbacs dataset](https://bioimage.io/#/?type=dataset&id=zero%2Fdataset_stardist_2d_deepbacs)


---

### Model fine-tunning via GUI

The Cellpose GUI allows you to manually correct segmentation outputs and to fine-tune the model to your specific dataset.

However, training new models on CPU is slow. For the sake of time, we will now intentionally train model just for few epochs (not enough to obtain a good model) purely to demonstrate the workflow. Later we will train better model leveraging GPU acceleration in Google Colab.

<div style="border-left: 4px solid #4CAF50; padding-left: 1em;">

**TASK**: Fine-tune a Cellpose model on the provided training dataset.

1. Open the **tricho_01.tif** image from the folder `cellpose_example/training_example`
2. Segment image using pre-trained model
3. Correct the segmentation and save it
4. Check the other already segmented **tricho** images
5. Explore the **Models > Train new model ...** menu
6. Fine-tune a pre-trained model on the provided dataset - set `n_epochs=10`) # We will use more epochs later in Google Colab
7. The custom model will be available in the dropdown menu 

</div>

- Tricho dataset - courtesy of Meri Eichner, PhD. - Algatech

---

### Cellpose in 3D

<div style="border-left: 4px solid #4CAF50; padding-left: 1em;">

**TASK**: Use Cellpose GUI to test the pre-trained built-in models on the example 3D image.

- Use the terminal with the course environment activated
- Start the Cellpose GUI for 3D datasets by running command: `cellpose --Zstack`
- Test the cyto3 model on the example image in the folder *cellpose_example/3d_example*
- Run cyto3 model with `stitch_threshold` parameter set to `> 0` and compare results

</div>

Data source: [Nuclei 3D image](https://scikit-image.org/docs/stable/api/skimage.data.html#skimage.data.cells3d)

---

## Programmatic use of Cellpose

How to use Cellpose within your Python code workflow.

<div style="border-left: 4px solid #4CAF50; padding-left: 1em;">

**TASK**: Running Cellpose in Python

1. From the cellpose library import the `io`, `models`, `core`, and `plot` modules
2. Initalize a pre-trained cellpose model using `models.Cellpose` and set argument for model selection to *nuclei* model - `model_type='nuclei'`
3. Load the *HPA_cell_1.tif* image using `io.imread`
4. Run inference using `.eval()` function on the model. This function returns 4 outputs: `masks, flows, styles, diams`
5. Visualize the mask results via `matplotlib.pyplot` or `cellpose.plot`
    - `cellpose.plot` have several functions for visualization of the results (e.g. `show_segmentation`)

</div>

NOTES:
- For the full list of available models, see the [Cellpose models documentation](https://cellpose.readthedocs.io/en/v3.1.1.1/models.html)
- By using `pretrained_model='/full/path/to/model'` you can load a local pre-trained model


In [ ]:
# Your code here



<div style="border-left: 4px solid #4CAF50; padding-left: 1em;">

**TASK**: Running Cellpose in batch

1. Initialize a pre-trained cellpose model using `model_type='cyto3'`
2. Load all images to be processed into a list using `io.imread` - read all `Deepbacs*` images
3. Evaluate the model on the image list - the `.eval` method. Returned values will now be *lists* of `masks, flows, styles and diams`
4. Visualize the results

</div>

In [ ]:
# Your code here

from pathlib import Path

folder_path = Path(r'set_path_to_data') # set correct path to folder with images
files = list(folder_path.glob('filename_pattern')) # set search pattern for Deepbacs images
images = ... # read images and store them as list




<details>
<summary><b>💡 Example solutions</b></summary>

```python

## ------------ Single rgb image - nuclei model ------------
from cellpose import io, models, core, plot
import matplotlib.pyplot as plt

# load image
image = io.imread(r"../data/cellpose_example/HPA_cell_1.tif")

# initialize model
model = models.Cellpose(model_type='nuclei')

# run prediction
masks_pred, flows, styles, diams = model.eval(image, diameter=60, channels=[2,0])

# visualize result
plt.figure(figsize=(8, 8))
plt.imshow(image[1, :, :], cmap="gray")
plt.imshow(masks_pred, alpha=0.4)
plt.title("Predicted masks")
plt.axis("off")
plt.show()

# or Cellpose.plot visualization
fig = plt.figure(figsize=(12, 5))
plot.show_segmentation(fig, image, masks_pred, flows[0])
plt.tight_layout()
plt.show()


## ------------ Batch ------------
from pathlib import Path

folder_path = Path(r'../data/cellpose_example')
files = list(folder_path.glob("Deep*.tif"))
images = [io.imread(img) for img in files]

model2 = models.Cellpose(model_type='cyto3')

masks, flows, styles, diams = model2.eval(images, diameter=None)

for i,m in zip(images, masks):
    plt.imshow(i, cmap='gray')
    plt.imshow(m, alpha = 0.4,cmap = 'nipy_spectral')
    plt.axis("off")
    plt.show()


```

</details>

---

### Model fine-tuning in Google Colab

Google Colab provides cloud-based Jupyter notebooks with free (limited) GPU access, making it possible to run Python code directly in the browser — no local installation needed. All you need is a Google account and an internet connection.
 
To enable GPU acceleration: **Runtime → Change runtime type → T4 GPU**
 
> NOTE: Interactive desktop GUIs (e.g. napari) are not supported in the Colab environment.


<div style="border-left: 4px solid #4CAF50; padding-left: 1em;">

**TASK**: Fine-tune Cellpose model from Google Colab

- Open a Cellpose fine-tuning notebook in Google Colab [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nunopimpaomartins/evolve-python-hybrid-course/tree/main/Day_3-Intro_Machine_and_Deep_Learning/notebooks/cellpose_finetune.ipynb) 
- Change runtime to GPU
- Explore the notebook together

</div>